# WeaveGuard AI — Étape C : moteur YOLO (détection supervisée)
**Dataset de référence : v2** (vérifié par hash). **Runtime Colab : GPU (T4 ou mieux)**.

Protocole (cahier des charges §7.1, AC-10) :
- 3 configurations comparées **sur la validation** : `v8n_640` (référence), `v8s_640`, `v8s_1024`.
- Choix du meilleur modèle sur la **validation** (mAP@50-95). Seuil de confiance calibré sur la validation (F1 maximal).
- Le **test est utilisé une seule fois**, pour le modèle choisi.
- Métriques par classe, matrice de confusion, latence, fausses alertes hors domaine (DTD), fiche modèle versionnée.

**Conseil** : lance d'abord uniquement `v8n_640` (≈ 30–40 min) pour valider la chaîne, puis ajoute les autres dans `RUN`. Chaque entraînement est sauvegardé sur Drive et reprend automatiquement si Colab se déconnecte.

In [ ]:
# 1) Drive, dépendances, GPU
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN')
assert torch.cuda.is_available(), 'Active un GPU : Exécution > Modifier le type d exécution > T4'

In [ ]:
# 2) Configuration
from pathlib import Path
import json, hashlib, shutil, random, zipfile, datetime, time
import numpy as np, pandas as pd, yaml
from PIL import Image, ImageDraw, ImageOps
import ultralytics
from ultralytics import YOLO

WG = Path('/content/drive/MyDrive/WeaveGuard')
DRIVE_OUT = WG / 'datasets'
MODELS = WG / 'models' / 'yolo'; MODELS.mkdir(parents=True, exist_ok=True)
V2 = Path('/content/weaveguard_dataset_v2')
SEED = 42
EPOCHS, PATIENCE = 100, 30
EXPERIMENTS = {
    'v8n_640':  dict(model='yolov8n.pt', imgsz=640,  batch=16),
    'v8s_640':  dict(model='yolov8s.pt', imgsz=640,  batch=16),
    'v8s_1024': dict(model='yolov8s.pt', imgsz=1024, batch=8),
}
RUN = ['v8n_640']          # <-- ajoute 'v8s_640', 'v8s_1024' une fois la chaîne validée
HYP = dict(mosaic=1.0, close_mosaic=10, scale=0.5, translate=0.1, fliplr=0.5, flipud=0.0,
           degrees=0.0, perspective=0.0, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3)
# degrees=0 et flipud=0 : une rotation de 90 degrés transformerait "Vertical LineMark" en "Horizontal Mark".
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)

In [ ]:
# 3) Dataset v2 en local + contrôle d'intégrité par hash
if not V2.exists():
    zipfile.ZipFile(DRIVE_OUT / 'weaveguard_dataset_v2.zip').extractall(V2)
stats = json.load(open(DRIVE_OUT / 'dataset_stats_v2.json'))
man = pd.read_csv(V2 / 'manifest.csv')
h = hashlib.sha256('\n'.join(sorted(f'{a}|{b}|{c}' for a, b, c in zip(man.name, man.md5, man.split))).encode()).hexdigest()
assert h == stats['dataset_hash_sha256'], 'Le dataset local ne correspond pas au v2 de référence !'
for s in ('train', 'val', 'test'):
    ni = len(list((V2 / 'images' / s).iterdir())); nl = len(list((V2 / 'labels' / s).iterdir()))
    assert ni == nl == stats['images_per_split'][s], (s, ni, nl)
    print(s, ni, 'images')
print('Intégrité OK, hash v2 =', h[:16], '...')
DATA = str(V2 / 'data.yaml'); print(open(DATA).read())

In [ ]:
# 4) Entraînement (reprise automatique) + évaluation sur la VALIDATION
def summarize(m):
    per = []
    for k, c in enumerate(m.box.ap_class_index):
        per.append(dict(cls=m.names[int(c)], P=float(m.box.p[k]), R=float(m.box.r[k]), F1=float(m.box.f1[k]),
                        AP50=float(m.box.ap50[k]), AP50_95=float(m.box.ap[k])))
    return dict(mAP50=float(m.box.map50), mAP50_95=float(m.box.map), mAP75=float(m.box.map75),
                P=float(m.box.mp), R=float(m.box.mr), per_class=per, speed_ms={k: float(v) for k, v in m.speed.items()})

def best_conf(m):
    try:
        f1 = np.asarray(m.box.f1_curve).mean(0); px = np.asarray(m.box.px)
        f1s = np.convolve(f1, np.ones(11) / 11, mode='same'); i = int(f1s.argmax())
        return float(px[i]), float(f1s[i])
    except Exception as e:
        print('[!] courbe F1 indisponible, conf=0.25 par défaut :', e); return 0.25, None

def train_one(name):
    exp = EXPERIMENTS[name]; d = MODELS / name; d.mkdir(parents=True, exist_ok=True)
    if (d / 'DONE.json').exists():
        print(f'[{name}] déjà terminé, ignoré'); return
    t0 = time.time(); last = d / 'weights' / 'last.pt'
    if last.exists():
        print(f'[{name}] reprise depuis last.pt'); model = YOLO(str(last)); model.train(resume=True)
    else:
        model = YOLO(exp['model'])
        model.train(data=DATA, epochs=EPOCHS, imgsz=exp['imgsz'], batch=exp['batch'], seed=SEED, patience=PATIENCE,
                    project=str(MODELS), name=name, exist_ok=True, cache='ram', workers=2, plots=True, **HYP)
    save_dir = Path(model.trainer.save_dir)
    if save_dir.resolve() != d.resolve(): shutil.copytree(save_dir, d, dirs_exist_ok=True)
    best = d / 'weights' / 'best.pt'; assert best.exists(), 'best.pt introuvable'
    m = YOLO(str(best)).val(data=DATA, split='val', imgsz=exp['imgsz'], batch=8, conf=0.001, iou=0.6, plots=True,
                            project=str(MODELS), name=name + '_val', exist_ok=True, verbose=False)
    res = summarize(m); res['conf_opt'], res['f1_at_conf_opt'] = best_conf(m)
    res.update(name=name, imgsz=exp['imgsz'], base_model=exp['model'], batch=exp['batch'], epochs_max=EPOCHS, seed=SEED,
               train_minutes=round((time.time() - t0) / 60, 1), weights_sha256=hashlib.sha256(best.read_bytes()).hexdigest(),
               dataset_hash=h, hyp=HYP, ultralytics=ultralytics.__version__, torch=torch.__version__, gpu=torch.cuda.get_device_name(0))
    json.dump(res, open(d / 'val_metrics.json', 'w'), indent=1)
    (d / 'DONE.json').write_text(json.dumps(dict(finished=datetime.datetime.now().isoformat(timespec='seconds'))))
    print(f"[{name}] val mAP50={res['mAP50']:.3f} mAP50-95={res['mAP50_95']:.3f} P={res['P']:.3f} R={res['R']:.3f} | conf_opt={res['conf_opt']:.3f}")

for name in RUN: train_one(name)

In [ ]:
# 5) Comparaison sur la VALIDATION et choix du meilleur modèle
rows, allres = [], {}
for name in EXPERIMENTS:
    f = MODELS / name / 'val_metrics.json'
    if f.exists():
        r = json.load(open(f)); allres[name] = r
        rows.append(dict(modele=name, imgsz=r['imgsz'], mAP50=round(r['mAP50'], 3), mAP50_95=round(r['mAP50_95'], 3),
                         P=round(r['P'], 3), R=round(r['R'], 3), conf_opt=round(r['conf_opt'], 3),
                         inference_ms=round(r['speed_ms'].get('inference', float('nan')), 1), minutes=r['train_minutes']))
cmp_df = pd.DataFrame(rows).sort_values('mAP50_95', ascending=False); display(cmp_df)
BEST = cmp_df.iloc[0].modele; print('>>> Meilleur sur la validation (mAP50-95) :', BEST)
if len(cmp_df) > 1:
    ref = 'v8n_640'
    if ref in allres: print(f"Écart vs référence {ref} : {allres[BEST]['mAP50_95'] - allres[ref]['mAP50_95']:+.3f} mAP50-95 (la validation est petite : écart < 0.02 = peu significatif)")
print('\nPar classe (validation) pour', BEST); display(pd.DataFrame(allres[BEST]['per_class']).round(3))

In [ ]:
# 6) Évaluation FINALE sur le TEST (une seule fois, modèle choisi sur la validation)
exp = EXPERIMENTS[BEST]; best_pt = MODELS / BEST / 'weights' / 'best.pt'; conf_opt = allres[BEST]['conf_opt']
tfile = MODELS / BEST / 'test_metrics.json'
if tfile.exists():
    print('Test déjà évalué pour ce modèle, résultats rechargés (le test ne se ré-utilise pas).'); test_res = json.load(open(tfile))
else:
    mdl = YOLO(str(best_pt))
    m1 = mdl.val(data=DATA, split='test', imgsz=exp['imgsz'], batch=8, conf=0.001, iou=0.6, plots=True,
                 project=str(MODELS), name=BEST + '_test', exist_ok=True, verbose=False)
    m2 = mdl.val(data=DATA, split='test', imgsz=exp['imgsz'], batch=8, conf=conf_opt, iou=0.6, plots=False, verbose=False)
    test_res = summarize(m1); test_res['at_conf_opt'] = dict(conf=conf_opt, P=float(m2.box.mp), R=float(m2.box.mr))
    test_res['n_images'] = stats['images_per_split']['test']
    json.dump(test_res, open(tfile, 'w'), indent=1)
print(f"TEST ({BEST}) : mAP50={test_res['mAP50']:.3f} mAP50-95={test_res['mAP50_95']:.3f}")
print(f"À conf={conf_opt:.3f} (calibrée sur val) : P={test_res['at_conf_opt']['P']:.3f} R={test_res['at_conf_opt']['R']:.3f}")
pc = pd.DataFrame(test_res['per_class'])
pc['images_test'] = pc.cls.map(lambda c: stats['images_per_class'][c]['test']); display(pc.round(3))
print('NB : classes avec peu d images en test => métriques fragiles (ex. Horizontal Mark : 10 images).')

In [ ]:
# 7) Fausses alertes hors domaine : échantillon de DTD (textures sans défaut textile annoté)
dtd_img = next((p for p in [WG / 'data' / 'dtd' / 'images', WG / 'data' / 'images'] if p.exists()), None)
dtd_res = None
if dtd_img is None:
    print('DTD introuvable, étape ignorée')
else:
    rg = random.Random(SEED); paths = []; cats = {}
    for c in sorted(d for d in dtd_img.iterdir() if d.is_dir()):
        fs = sorted(f for f in c.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}); pick = rg.sample(fs, min(6, len(fs)))
        paths += pick; cats.update({str(p): c.name for p in pick})
    mdl = YOLO(str(best_pt)); out = mdl.predict(source=[str(p) for p in paths], conf=conf_opt, imgsz=exp['imgsz'], verbose=False)
    det_per_cat = {}
    for p, r in zip(paths, out): det_per_cat.setdefault(cats[str(p)], []).append(len(r.boxes))
    n_with = sum(1 for r in out if len(r.boxes) > 0)
    dtd_res = dict(n_images=len(paths), images_with_detection=n_with, rate=round(n_with / len(paths), 3), conf=conf_opt,
                   top_categories={k: round(float(np.mean([x > 0 for x in v])), 2) for k, v in sorted(det_per_cat.items(), key=lambda kv: -np.mean([x > 0 for x in kv[1]]))[:8]})
    print(f"DTD : {n_with}/{len(paths)} images avec au moins une détection ({dtd_res['rate']:.1%}) à conf={conf_opt:.3f}")
    print('Catégories les plus déclenchées :', dtd_res['top_categories'])
    print('NB : domaine différent, et certaines catégories (perforated, pitted, stained...) peuvent ressembler à de vrais défauts. Indicatif seulement.')

In [ ]:
# 8) Analyse qualitative sur la VALIDATION : vérité terrain (gauche) vs prédictions (droite)
import matplotlib.pyplot as plt
mdl = YOLO(str(best_pt)); names = mdl.names
vimgs = sorted((V2 / 'images' / 'val').iterdir()); sample = random.Random(SEED).sample(vimgs, 6)
fig, axes = plt.subplots(6, 2, figsize=(12, 30))
for k, p in enumerate(sample):
    im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); W, H = im.size
    gt = im.copy(); dr = ImageDraw.Draw(gt)
    for line in (V2 / 'labels' / 'val' / (p.stem + '.txt')).read_text().splitlines():
        s = line.split()
        if len(s) == 5:
            c = int(s[0]); cx, cy, bw, bh = map(float, s[1:])
            dr.rectangle([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H], outline='lime', width=max(3, W // 250))
            dr.text(((cx-bw/2)*W + 4, (cy-bh/2)*H + 4), names[c], fill='lime')
    r = mdl.predict(source=str(p), conf=conf_opt, imgsz=exp['imgsz'], verbose=False)[0]
    axes[k, 0].imshow(gt); axes[k, 0].set_title('vérité terrain', fontsize=9); axes[k, 0].axis('off')
    axes[k, 1].imshow(r.plot()[:, :, ::-1]); axes[k, 1].set_title(f'prédictions (conf >= {conf_opt:.2f})', fontsize=9); axes[k, 1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 9) Fiche modèle + export versionné, rapport à coller dans le chat
final_dir = WG / 'models' / 'weaveguard_yolo_v1'; final_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt, final_dir / 'best.pt')
for f in ('results.csv', 'args.yaml', 'results.png', 'confusion_matrix_normalized.png', 'confusion_matrix.png'):
    for base in (MODELS / BEST, MODELS / (BEST + '_test'), MODELS / (BEST + '_val')):
        if (base / f).exists(): shutil.copy2(base / f, final_dir / f'{base.name}_{f}'); break
card = dict(
    model_id='weaveguard_yolo_v1', role='Détection supervisée de défauts textiles (classes connues uniquement)',
    classes=stats['classes'], selected_experiment=BEST, base_model=allres[BEST]['base_model'], imgsz=allres[BEST]['imgsz'],
    weights_sha256=allres[BEST]['weights_sha256'], dataset_version='v2', dataset_hash=h, seed=SEED,
    conf_threshold=dict(value=conf_opt, rule='F1 macro maximal sur la validation (lissé)', iou_nms=0.6),
    validation={k: allres[BEST][k] for k in ('mAP50', 'mAP50_95', 'P', 'R')}, validation_per_class=allres[BEST]['per_class'],
    test=dict(mAP50=test_res['mAP50'], mAP50_95=test_res['mAP50_95'], at_conf_opt=test_res['at_conf_opt'],
              n_images=test_res['n_images'], per_class=test_res['per_class']),
    speed_ms_colab_gpu=allres[BEST]['speed_ms'], gpu=allres[BEST]['gpu'], dtd_out_of_domain=dtd_res,
    hyp=HYP, epochs_max=EPOCHS, ultralytics=ultralytics.__version__, torch=torch.__version__,
    comparison_val={r['modele']: dict(mAP50=r['mAP50'], mAP50_95=r['mAP50_95']) for r in rows},
    limits=['9 classes seulement ; toute autre catégorie de défaut est hors périmètre',
            'test de 150 images, 10 à 29 images par classe : métriques à intervalle large',
            'pas de probabilité calibrée : le score est une confiance du modèle, pas une probabilité',
            'latence mesurée sur GPU Colab ; à remesurer sur le matériel cible (GTX 1050)'],
    created=datetime.datetime.now().isoformat(timespec='seconds'))
(final_dir / 'model_card.json').write_text(json.dumps(card, indent=1, ensure_ascii=False), encoding='utf-8')
print('Modèle exporté :', final_dir); print('=' * 70)
print(json.dumps({k: card[k] for k in ('selected_experiment', 'imgsz', 'conf_threshold', 'validation', 'test', 'comparison_val', 'speed_ms_colab_gpu', 'dtd_out_of_domain')}, indent=1, ensure_ascii=False, default=str))